In [1]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    scikit-learn \
    pandas \
    numpy \
    scipy \
    huggingface_hub \
    evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [2]:
import os
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from scipy.stats import pearsonr, spearmanr

from torch import nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "answerdotai/ModernBERT-base"

MAX_LENGTH = 512

OUTPUT_DIR = "./promptforge-quality"

NUM_LABELS = 7

LABEL_NAMES = [
    "clarity",
    "specificity",
    "context",
    "goal_definition",
    "constraints",
    "completeness",
    "actionability",
]

print("Model:", MODEL_NAME)
print("Labels:", LABEL_NAMES)

Model: answerdotai/ModernBERT-base
Labels: ['clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability']


In [4]:
import random

TASKS = {
    "coding": [
        "Build a REST API",
        "Create a React application",
        "Write a Python script",
        "Build a CLI tool",
        "Create a database schema",
    ],
    "writing": [
        "Write a blog post",
        "Write an email",
        "Write a technical article",
        "Create a product description",
        "Write a LinkedIn post",
    ],
    "research": [
        "Research a technology",
        "Compare two databases",
        "Analyze a software architecture",
        "Explain a machine learning technique",
        "Evaluate a programming language",
    ],
    "data": [
        "Analyze a dataset",
        "Create a data visualization",
        "Build a machine learning model",
        "Clean a dataset",
        "Generate a statistical report",
    ],
    "creative": [
        "Create a story",
        "Write a game concept",
        "Design a character",
        "Create a marketing campaign",
        "Generate a product idea",
    ],
}

AUDIENCES = [
    "beginner developers",
    "experienced developers",
    "software engineers",
    "students",
    "technical managers",
    "startup founders",
    "data scientists",
    "general users",
]

CONSTRAINTS = [
    "Use Python 3.12.",
    "Use TypeScript and React.",
    "Keep the solution under 200 lines.",
    "Return the answer as Markdown.",
    "Include complete working code.",
    "Do not use external libraries.",
    "Include error handling.",
    "Make the solution production-ready.",
    "Use PostgreSQL.",
    "Make it mobile responsive.",
]

OUTPUT_FORMATS = [
    "Return the answer as a numbered list.",
    "Return complete source code.",
    "Return JSON.",
    "Return a step-by-step explanation.",
    "Return a Markdown document.",
    "Include examples.",
]

CONTEXTS = [
    "This is for a university project.",
    "This is for a production SaaS application.",
    "This is for an internal developer tool.",
    "This will be used by beginners.",
    "This is a prototype for a startup.",
    "This will run locally on a laptop.",
]

In [5]:
def clamp(value):
    return max(0, min(100, int(round(value))))


def generate_example():
    domain = random.choice(list(TASKS.keys()))
    task = random.choice(TASKS[domain])

    level = random.randint(0, 4)

    clarity = 30 + level * 17 + random.randint(-5, 5)
    specificity = 15 + level * 20 + random.randint(-5, 5)
    context = 10 + level * 20 + random.randint(-5, 5)
    goal_definition = 25 + level * 17 + random.randint(-5, 5)
    constraints = 5 + level * 22 + random.randint(-5, 5)
    completeness = 15 + level * 20 + random.randint(-5, 5)
    actionability = 20 + level * 18 + random.randint(-5, 5)

    clarity = clamp(clarity)
    specificity = clamp(specificity)
    context = clamp(context)
    goal_definition = clamp(goal_definition)
    constraints = clamp(constraints)
    completeness = clamp(completeness)
    actionability = clamp(actionability)

    parts = []

    if level == 0:
        prompt = random.choice([
            "Make an app.",
            "Build something.",
            "Write something good.",
            "Help me with this.",
            "Create a website.",
            "Make this better.",
            "Analyze this.",
            "Build me a project.",
        ])

    elif level == 1:
        prompt = f"{task}."

    elif level == 2:
        prompt = f"{task} for {random.choice(AUDIENCES)}."

    elif level == 3:
        prompt = (
            f"{task} for {random.choice(AUDIENCES)}. "
            f"{random.choice(CONTEXTS)} "
            f"{random.choice(OUTPUT_FORMATS)}"
        )

    else:
        prompt = (
            f"{task} for {random.choice(AUDIENCES)}. "
            f"{random.choice(CONTEXTS)} "
            f"Requirements: {random.choice(CONSTRAINTS)} "
            f"{random.choice(CONSTRAINTS)} "
            f"{random.choice(OUTPUT_FORMATS)} "
            f"Explain important design decisions and include examples."
        )

    # Make labels correlate with actual prompt complexity.
    overall = np.mean([
        clarity,
        specificity,
        context,
        goal_definition,
        constraints,
        completeness,
        actionability,
    ])

    return {
        "prompt": prompt,
        "clarity": clarity,
        "specificity": specificity,
        "context": context,
        "goal_definition": goal_definition,
        "constraints": constraints,
        "completeness": completeness,
        "actionability": actionability,
        "quality_score": clamp(overall),
        "task_type": domain,
    }

In [6]:
NUM_EXAMPLES = 25000

examples = [
    generate_example()
    for _ in range(NUM_EXAMPLES)
]

df = pd.DataFrame(examples)

print("Dataset size:", len(df))
print()
print(df.head())

Dataset size: 25000

                                              prompt  clarity  specificity  \
0              Build a REST API for data scientists.       62           53   
1                                      Analyze this.       28           13   
2  Create a product description for students. Thi...       97           90   
3  Write a Python script for experienced developers.       68           54   
4  Design a character for technical managers. Thi...       98           99   

   context  goal_definition  constraints  completeness  actionability  \
0       47               55           54            58             52   
1       13               29            0            18             18   
2       87               94           93            94             89   
3       45               61           52            51             57   
4       88               89           88           100             90   

   quality_score task_type  
0             54    coding  
1            

In [7]:
NUM_EXAMPLES = 25000

examples = [
    generate_example()
    for _ in range(NUM_EXAMPLES)
]

df = pd.DataFrame(examples)

print("Dataset size:", len(df))
print()
print(df.head())

Dataset size: 25000

                                              prompt  clarity  specificity  \
0                              Write something good.       28           16   
1  Explain a machine learning technique for exper...       95          100   
2  Compare two databases for software engineers. ...       82           76   
3      Analyze a software architecture for students.       61           52   
4                             Compare two databases.       42           30   

   context  goal_definition  constraints  completeness  actionability  \
0       11               29            6            12             23   
1       95               93           98           100             88   
2       74               78           68            70             77   
3       54               57           46            52             55   
4       32               42           32            31             39   

   quality_score task_type  
0             18    coding  
1            

In [8]:
print(df["quality_score"].describe())

print("\nTask distribution:")
print(df["task_type"].value_counts())

print("\nQuality distribution:")
print(
    pd.cut(
        df["quality_score"],
        bins=[0, 20, 40, 60, 80, 100],
        labels=["0-20", "21-40", "41-60", "61-80", "81-100"]
    ).value_counts().sort_index()
)

count    25000.000000
mean        55.375880
std         27.059591
min         13.000000
25%         35.000000
50%         55.000000
75%         75.000000
max         98.000000
Name: quality_score, dtype: float64

Task distribution:
task_type
research    5143
creative    5023
data        4971
coding      4953
writing     4910
Name: count, dtype: int64

Quality distribution:
quality_score
0-20      4988
21-40     5019
41-60     5077
61-80     4893
81-100    5023
Name: count, dtype: int64


In [9]:
DATASET_PATH = "/content/promptforge_dataset.csv"

df.to_csv(DATASET_PATH, index=False)

print("Saved:", DATASET_PATH)

Saved: /content/promptforge_dataset.csv


In [10]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 20000
Validation: 2500
Test: 2500


In [11]:
train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'task_type'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['prompt', 'clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'task_type'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['prompt', 'clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'task_type'],
        num_rows: 2500
    })
})

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Tokenizer loaded.


In [13]:
def tokenize_function(batch):
    return tokenizer(
        batch["prompt"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=[
        "prompt",
        "task_type",
    ],
)

tokenized_dataset

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'input_ids', 'attention_mask'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['clarity', 'specificity', 'context', 'goal_definition', 'constraints', 'completeness', 'actionability', 'quality_score', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
})

In [14]:
label_columns = [
    "clarity",
    "specificity",
    "context",
    "goal_definition",
    "constraints",
    "completeness",
    "actionability",
]

def add_labels(example):
    example["labels"] = [
        float(example[col])
        for col in label_columns
    ]

    return example


tokenized_dataset = tokenized_dataset.map(
    add_labels
)

tokenized_dataset = tokenized_dataset.remove_columns(
    label_columns + ["quality_score"]
)

print(tokenized_dataset["train"][0])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

{'input_ids': [50281, 40731, 247, 1885, 2934, 323, 20500, 33663, 15, 831, 310, 323, 247, 3275, 322, 5781, 52, 2898, 15, 37880, 942, 27, 17105, 2228, 10885, 15, 16600, 253, 2900, 762, 1052, 3104, 15, 17105, 6667, 15, 14499, 404, 1774, 2216, 7089, 285, 2486, 6667, 15, 50282], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [100.0, 91.0, 90.0, 93.0, 94.0, 94.0, 96.0]}


In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

In [16]:
class PromptForgeQualityModel(nn.Module):

    def __init__(
        self,
        model_name,
        num_labels=7,
        dropout=0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.quality_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

        self.dimension_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels),
        )

    def mean_pooling(
        self,
        last_hidden_state,
        attention_mask,
    ):
        mask = attention_mask.unsqueeze(-1).expand(
            last_hidden_state.size()
        ).float()

        summed = torch.sum(
            last_hidden_state * mask,
            dim=1
        )

        counts = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        return summed / counts

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        **kwargs,
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled = self.mean_pooling(
            outputs.last_hidden_state,
            attention_mask,
        )

        pooled = self.dropout(pooled)

        dimension_logits = self.dimension_head(
            pooled
        )

        quality_logit = self.quality_head(
            pooled
        )

        dimension_predictions = (
            torch.sigmoid(dimension_logits) * 100
        )

        quality_prediction = (
            torch.sigmoid(quality_logit) * 100
        )

        loss = None

        if labels is not None:

            dimension_targets = labels

            target_quality = (
                dimension_targets.mean(
                    dim=1,
                    keepdim=True
                )
            )

            dimension_loss = F.mse_loss(
                dimension_predictions,
                dimension_targets
            )

            quality_loss = F.mse_loss(
                quality_prediction,
                target_quality
            )

            loss = (
                0.8 * dimension_loss
                +
                0.2 * quality_loss
            )

        return {
            "loss": loss,
            "logits": dimension_predictions,
            "quality": quality_prediction,
        }

In [17]:
model = PromptForgeQualityModel(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Device:", device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [18]:
class PromptForgeTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):

        labels = inputs.pop("labels")

        outputs = model(
            **inputs,
            labels=labels,
        )

        loss = outputs["loss"]

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [19]:
def compute_metrics(eval_prediction):

    predictions, labels = eval_prediction

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    labels = np.array(labels)

    predictions = np.clip(
        predictions,
        0,
        100,
    )

    mae = mean_absolute_error(
        labels,
        predictions,
    )

    rmse = np.sqrt(
        mean_squared_error(
            labels,
            predictions,
        )
    )

    correlations = []

    for i in range(labels.shape[1]):

        try:
            corr, _ = pearsonr(
                labels[:, i],
                predictions[:, i],
            )

            correlations.append(corr)

        except Exception:
            correlations.append(0.0)

    mean_pearson = np.mean(
        correlations
    )

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "pearson": float(mean_pearson),
    }

In [21]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,

    warmup_steps=500,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="pearson",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none",

    remove_unused_columns=False,
)

In [23]:
trainer = PromptForgeTrainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

In [24]:
train_result = trainer.train()

print(train_result)

Step,Training Loss,Validation Loss,Mae,Rmse,Pearson
500,26.644387,9.871017,2.884492,3.396830,0.992504
1000,21.224734,9.126569,2.826324,3.303577,0.992597
1500,19.886510,8.565609,2.769750,3.211330,0.992971
2000,19.875010,8.764217,2.783758,3.234224,0.993007
2500,19.067690,8.504514,2.761932,3.198482,0.993088
3000,19.221814,8.417958,2.755157,3.183390,0.993080
3500,18.686802,8.279978,2.741331,3.161761,0.993091
3750,18.642869,8.236128,2.733622,3.153974,0.993113


TrainOutput(global_step=3750, training_loss=55.552009977213544, metrics={'train_runtime': 957.2752, 'train_samples_per_second': 62.678, 'train_steps_per_second': 3.917, 'total_flos': 0.0, 'train_loss': 55.552009977213544, 'epoch': 3.0})


In [25]:
evaluation = trainer.evaluate(
    tokenized_dataset["test"]
)

evaluation

Training Loss,Validation Loss,Step,Mae,Rmse,Pearson
18.642869,8.227754,3750,2.719055,3.150229,0.993039


{'eval_loss': 8.227753639221191,
 'eval_mae': 2.71905517578125,
 'eval_rmse': 3.150228667663561,
 'eval_pearson': 0.9930385947227478}

In [27]:
predictions_output = trainer.predict(
    tokenized_dataset["test"]
)

raw_predictions = predictions_output.predictions
labels = predictions_output.label_ids

print("Prediction type:", type(raw_predictions))

if isinstance(raw_predictions, tuple):
    print("Number of prediction outputs:", len(raw_predictions))

    # Our model returns:
    # [dimension_predictions, quality_prediction]
    predictions = raw_predictions[0]
else:
    predictions = raw_predictions

predictions = np.asarray(predictions)
labels = np.asarray(labels)

print("Predictions shape:", predictions.shape)
print("Labels shape:", labels.shape)

predictions = np.clip(
    predictions,
    0,
    100,
)

for i, name in enumerate(LABEL_NAMES):

    mae = mean_absolute_error(
        labels[:, i],
        predictions[:, i],
    )

    rmse = np.sqrt(
        mean_squared_error(
            labels[:, i],
            predictions[:, i],
        )
    )

    pearson = pearsonr(
        labels[:, i],
        predictions[:, i],
    )[0]

    spearman = spearmanr(
        labels[:, i],
        predictions[:, i],
    )[0]

    print(f"\n{name}")
    print("MAE:", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("Pearson:", round(pearson, 3))
    print("Spearman:", round(spearman, 3))

Prediction type: <class 'tuple'>
Number of prediction outputs: 2
Predictions shape: (2500, 7)
Labels shape: (2500, 7)

clarity
MAE: 2.648
RMSE: 3.067
Pearson: 0.992
Spearman: 0.96

specificity
MAE: 2.716
RMSE: 3.151
Pearson: 0.994
Spearman: 0.961

context
MAE: 2.694
RMSE: 3.131
Pearson: 0.994
Spearman: 0.96

goal_definition
MAE: 2.721
RMSE: 3.156
Pearson: 0.991
Spearman: 0.961

constraints
MAE: 2.755
RMSE: 3.194
Pearson: 0.995
Spearman: 0.959

completeness
MAE: 2.761
RMSE: 3.191
Pearson: 0.994
Spearman: 0.959

actionability
MAE: 2.738
RMSE: 3.159
Pearson: 0.992
Spearman: 0.96


In [28]:
true_overall = labels.mean(axis=1)
pred_overall = predictions.mean(axis=1)

print(
    "Overall MAE:",
    mean_absolute_error(
        true_overall,
        pred_overall
    )
)

print(
    "Overall RMSE:",
    np.sqrt(
        mean_squared_error(
            true_overall,
            pred_overall
        )
    )
)

print(
    "Overall Pearson:",
    pearsonr(
        true_overall,
        pred_overall
    )[0]
)

print(
    "Overall Spearman:",
    spearmanr(
        true_overall,
        pred_overall
    )[0]
)

Overall MAE: 0.9617130756378174
Overall RMSE: 1.2018364781213489
Overall Pearson: 0.99899924
Overall Spearman: 0.9577865854467139


In [29]:
def score_prompt(
    prompt,
    model=model,
    tokenizer=tokenizer,
    device=device,
):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(
            **inputs
        )

    scores = (
        outputs["logits"]
        .squeeze(0)
        .detach()
        .cpu()
        .numpy()
    )

    overall = float(
        scores.mean()
    )

    return {
        "quality_score": round(
            overall,
            2
        ),
        "dimensions": {
            name: round(
                float(score),
                2
            )
            for name, score
            in zip(
                LABEL_NAMES,
                scores
            )
        },
    }

In [30]:
prompt = """
Build a modern SaaS application for developers.
Use Next.js, TypeScript and PostgreSQL.
Include authentication, billing, an admin dashboard,
API documentation and responsive design.
Return the complete project structure and implementation.
"""

result = score_prompt(prompt)

print(
    json.dumps(
        result,
        indent=2
    )
)

{
  "quality_score": 90.36,
  "dimensions": {
    "clarity": 95.0,
    "specificity": 91.38,
    "context": 85.25,
    "goal_definition": 90.75,
    "constraints": 89.38,
    "completeness": 91.62,
    "actionability": 89.12
  }
}


In [31]:
test_prompts = [
    "Make an app.",
    "Build me a website.",
    "Write something about AI.",
    "Make a Python API for beginners.",
    """
    Build a production-ready REST API using FastAPI and PostgreSQL.
    Implement JWT authentication, request validation, structured
    error handling and OpenAPI documentation. The API will be used
    by a React frontend. Return the complete project structure,
    implementation and example requests.
    """,
]

for prompt in test_prompts:

    result = score_prompt(prompt)

    print("=" * 80)

    print("PROMPT:")
    print(prompt.strip())

    print("\nSCORE:")
    print(result["quality_score"])

    print("\nDIMENSIONS:")

    for key, value in result["dimensions"].items():
        print(f"{key:20s}: {value}")

PROMPT:
Make an app.

SCORE:
17.35

DIMENSIONS:
clarity             : 30.03
specificity         : 15.28
context             : 10.12
goal_definition     : 25.31
constraints         : 5.35
completeness        : 14.98
actionability       : 20.39
PROMPT:
Build me a website.

SCORE:
17.06

DIMENSIONS:
clarity             : 29.88
specificity         : 14.89
context             : 9.98
goal_definition     : 24.89
constraints         : 5.02
completeness        : 14.78
actionability       : 20.02
PROMPT:
Write something about AI.

SCORE:
24.57

DIMENSIONS:
clarity             : 37.38
specificity         : 22.64
context             : 17.31
goal_definition     : 32.56
constraints         : 12.37
completeness        : 22.47
actionability       : 27.25
PROMPT:
Make a Python API for beginners.

SCORE:
55.81

DIMENSIONS:
clarity             : 64.25
specificity         : 55.56
context             : 50.34
goal_definition     : 59.72
constraints         : 49.81
completeness        : 55.38
actionability  

In [32]:
MODEL_OUTPUT = "/content/promptforge-quality-model"

os.makedirs(
    MODEL_OUTPUT,
    exist_ok=True
)

torch.save(
    model.state_dict(),
    os.path.join(
        MODEL_OUTPUT,
        "pytorch_model.bin"
    )
)

tokenizer.save_pretrained(
    MODEL_OUTPUT
)

print(
    "Model saved to:",
    MODEL_OUTPUT
)

Model saved to: /content/promptforge-quality-model
